In [ ]:

# FitAI — Workout Dataset Pipeline (Notebook 1)

# CELL 1 — Install dependencies
!pip install google-generativeai google-api-python-client google-auth-httplib2 google-auth-oauthlib tqdm pandas --quiet

# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/FitAI/datasets/workouts'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Saving to: {SAVE_DIR}")

# CELL 3 — API Key Configuration
from google.colab import userdata
import google.generativeai as genai
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
YOUTUBE_API_KEY = userdata.get('YOUTUBE_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemma-3-27b-it')
print("Gemma 4 ready")

# CELL 4 — Discipline Search Queries
SEARCH_QUERIES = [
  "bodybuilding workout tutorial", "bodybuilding chest day routine", "bodybuilding leg day training",
  "calisthenics beginner routine", "calisthenics advanced skills tutorial", "calisthenics full body workout",
  "martial arts conditioning workout", "MMA strength training", "boxing fitness workout",
  "tai chi for beginners tutorial", "tai chi full routine", "tai chi balance and flexibility",
  "weight loss cardio workout", "fat burning HIIT workout", "weight loss full body routine",
  "general fitness workout beginner", "full body strength training", "functional fitness workout",
  "yoga full body routine", "yoga for beginners", "yoga strength and flexibility",
  "soccer conditioning training", "soccer agility drills", "soccer speed and endurance training",
  "basketball strength training", "basketball agility drills", "basketball conditioning workout",
  "rugby conditioning workout", "rugby strength training", "rugby power and speed drills",
  "swimming dryland training", "swim strength conditioning", "swimming endurance workout",
  "tennis footwork drills", "tennis conditioning workout", "tennis agility and speed training",
  "sprint training workout", "athletics conditioning drills", "track and field strength training",
]
VIDEOS_PER_QUERY = 15
print(f"Total queries: {len(SEARCH_QUERIES)}")
print(f"Estimated total videos: {len(SEARCH_QUERIES) * VIDEOS_PER_QUERY}")

# CELL 5 — YouTube Search Function
from googleapiclient.discovery import build
import time

youtube = build('youtube', 'v3', developerKey=YOUTUBE_API_KEY)

def search_youtube(query, max_results=15):
    try:
        response = youtube.search().list(
            q=query, part='snippet', maxResults=max_results,
            type='video', videoDuration='medium', relevanceLanguage='en'
        ).execute()
        videos = []
        for item in response.get('items', []):
            videos.append({
                'video_id': item['id']['videoId'],
                'url': f"https://www.youtube.com/watch?v={item['id']['videoId']}",
                'title': item['snippet']['title'],
                'channel': item['snippet']['channelTitle'],
                'query': query
            })
        return videos
    except Exception as e:
        print(f"Search error for '{query}': {e}")
        return []

print("YouTube search function ready")

# CELL 6 — Gemma 4 Extraction Function
import json
import re

EXTRACTION_PROMPT = """
You are a fitness data extraction expert. Analyze this YouTube workout video and extract structured data.

Video URL: {url}
Video Title: {title}
Search Query Used: {query}

Extract ALL exercises shown or described in this video. For each exercise return a JSON array where each item has:
- exercise_name: string
- discipline: one of [bodybuilding, calisthenics, martial_arts, tai_chi, weight_loss, general_fitness, yoga, soccer, basketball, rugby, swimming, tennis, athletics]
- primary_muscles: array of muscle group names
- secondary_muscles: array of muscle group names
- equipment_needed: array (use "none" for bodyweight)
- difficulty_level: one of [beginner, intermediate, advanced]
- environment: one of [home, gym, outdoor, both]
- sets_recommended: string (e.g. "3-4")
- reps_recommended: string (e.g. "8-12" or "30 seconds")
- rest_time_seconds: number
- form_cues: array of short coaching tips
- sport_specific: boolean
- sport: string or null

Return ONLY a valid JSON array. No explanation, no markdown, no preamble.
"""

def extract_with_gemma(video):
    prompt = EXTRACTION_PROMPT.format(
        url=video['url'], title=video['title'], query=video['query']
    )
    try:
        response = model.generate_content(prompt)
        raw = response.text.strip()
        raw = re.sub(r'```json|```', '', raw).strip()
        exercises = json.loads(raw)
        for ex in exercises:
            ex['source_video_id'] = video['video_id']
            ex['source_title'] = video['title']
            ex['source_channel'] = video['channel']
        return exercises
    except json.JSONDecodeError:
        print(f"  JSON parse error for: {video['title'][:50]}")
        return []
    except Exception as e:
        print(f"  Gemma error: {e}")
        return []

print("Gemma extraction function ready")

# CELL 7 — Main Pipeline Loop
from tqdm import tqdm
import pandas as pd

all_exercises = []
all_videos = []
seen_video_ids = set()
failed_videos = []

print("Starting pipeline...\n")

for query in tqdm(SEARCH_QUERIES, desc="Searching disciplines"):
    print(f"\n Searching: {query}")
    videos = search_youtube(query, VIDEOS_PER_QUERY)
    print(f"  Found {len(videos)} videos")
    for video in videos:
        if video['video_id'] in seen_video_ids:
            continue
        seen_video_ids.add(video['video_id'])
        all_videos.append(video)
        print(f"  Extracting: {video['title'][:60]}")
        exercises = extract_with_gemma(video)
        if exercises:
            all_exercises.extend(exercises)
            print(f"  Extracted {len(exercises)} exercises")
        else:
            failed_videos.append(video)
        time.sleep(4)
    checkpoint_path = f"{SAVE_DIR}/checkpoint_exercises.json"
    with open(checkpoint_path, 'w') as f:
        json.dump(all_exercises, f, indent=2)

print(f"\nPipeline complete!")
print(f"Total videos processed: {len(seen_video_ids)}")
print(f"Total exercises extracted: {len(all_exercises)}")
print(f"Failed videos: {len(failed_videos)}")

# CELL 8 — Build Binary Matching Dataset
ALL_MUSCLE_GROUPS = [
    'chest', 'back', 'shoulders', 'biceps', 'triceps', 'forearms',
    'core', 'abs', 'obliques', 'quads', 'hamstrings', 'glutes',
    'calves', 'hip_flexors', 'adductors', 'full_body', 'cardiovascular'
]

binary_records = []
for ex in all_exercises:
    record = {
        'exercise_name': ex.get('exercise_name', ''),
        'discipline': ex.get('discipline', ''),
        'difficulty_level': ex.get('difficulty_level', ''),
        'environment': ex.get('environment', ''),
        'equipment_needed': ', '.join(ex.get('equipment_needed', [])),
        'sets_recommended': ex.get('sets_recommended', ''),
        'reps_recommended': ex.get('reps_recommended', ''),
        'rest_time_seconds': ex.get('rest_time_seconds', 60),
        'form_cues': ' | '.join(ex.get('form_cues', [])),
        'sport_specific': ex.get('sport_specific', False),
        'sport': ex.get('sport', ''),
        'source_video_id': ex.get('source_video_id', ''),
    }
    primary = [m.lower() for m in ex.get('primary_muscles', [])]
    secondary = [m.lower() for m in ex.get('secondary_muscles', [])]
    for muscle in ALL_MUSCLE_GROUPS:
        record[f'primary_{muscle}'] = 1 if muscle in primary else 0
        record[f'secondary_{muscle}'] = 1 if muscle in secondary else 0
    binary_records.append(record)

df = pd.DataFrame(binary_records)
df = df.drop_duplicates(subset=['exercise_name', 'discipline'])
print(f"Binary dataset shape: {df.shape}")

# CELL 9 — Build Fine-tuning Dataset
finetune_records = []
for ex in all_exercises:
    finetune_records.append({
        "instruction": "Explain how to perform this exercise correctly.",
        "input": ex.get('exercise_name', ''),
        "output": f"Here is how to perform {ex.get('exercise_name', '')}: " + " ".join(ex.get('form_cues', []))
    })
    finetune_records.append({
        "instruction": "What muscles does this exercise target?",
        "input": ex.get('exercise_name', ''),
        "output": f"Primary muscles: {', '.join(ex.get('primary_muscles', []))}. Secondary muscles: {', '.join(ex.get('secondary_muscles', []))}."
    })
    finetune_records.append({
        "instruction": "Recommend sets, reps, and rest time for this exercise.",
        "input": f"{ex.get('exercise_name', '')} for {ex.get('difficulty_level', '')} level",
        "output": f"Perform {ex.get('sets_recommended', '3')} sets of {ex.get('reps_recommended', '10')} reps with {ex.get('rest_time_seconds', 60)} seconds rest between sets."
    })
print(f"Fine-tuning records generated: {len(finetune_records)}")

# CELL 10 — Save All Datasets to Google Drive
raw_path = f"{SAVE_DIR}/raw_exercises.json"
with open(raw_path, 'w') as f:
    json.dump(all_exercises, f, indent=2)
print(f"Raw exercises saved: {raw_path}")

csv_path = f"{SAVE_DIR}/binary_exercise_muscle_dataset.csv"
df.to_csv(csv_path, index=False)
print(f"Binary dataset saved: {csv_path}")

ft_path = f"{SAVE_DIR}/finetune_workout_dataset.json"
with open(ft_path, 'w') as f:
    json.dump(finetune_records, f, indent=2)
print(f"Fine-tuning dataset saved: {ft_path}")

fail_path = f"{SAVE_DIR}/failed_videos.json"
with open(fail_path, 'w') as f:
    json.dump(failed_videos, f, indent=2)

print(f"\nAll files saved to Google Drive!")
print(f"Raw exercises:      {len(all_exercises)} records")
print(f"Binary dataset:     {len(df)} unique exercises")
print(f"Fine-tuning pairs:  {len(finetune_records)} records")
print(f"Failed videos:      {len(failed_videos)} videos")
